# Libs

In [ ]:
# Install necessary libraries
!pip install pandas editdistance matplotlib librosa

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import librosa
import urllib.request
import pandas as pd
import numpy as np
import time
import os
import json
import tarfile
from typing import Dict, List, Tuple
import editdistance
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [2]:
PAD_ID = 0
SOS_ID = 1
EOS_ID = 2
UNK_ID = 3
BLANK_ID = PAD_ID
SAMPLE_RATE = 16000
INPUT_FEATURE_DIM = 120 # 40 Mel + 40 Delta + 40 Double Delta

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


# Data

In [3]:
# DATA_URL = "http://www.openslr.org/resources/12/dev-clean.tar.gz"
# DATA_FOLDER = 'data'
# os.makedirs(DATA_FOLDER, exist_ok=True)
# tar_path = os.path.join(DATA_FOLDER, 'dev-clean.tar.gz')

# if not os.path.exists(os.path.join(DATA_FOLDER, 'LibriSpeech/dev-clean')):
#     print("Downloading LibriSpeech dev-clean (approx 340MB)...")
#     try:
#         if not os.path.exists(tar_path):
#             print(f"Downloading {DATA_URL}...")
#             req = urllib.request.Request(DATA_URL, headers={'User-Agent': 'Mozilla/5.0'})
#             with urllib.request.urlopen(req) as response:
#                 with open(tar_path, 'wb') as outfile:
#                     outfile.write(response.read())
#             print(f"Downloaded to {tar_path}")

#         with tarfile.open(tar_path, 'r:gz') as tar:
#             print("Extracting files...")
#             tar.extractall(path=DATA_FOLDER)
#             print("Extraction complete.")

#     except Exception as e:
#         print(f"Error during download or extraction: {e}. Check network connection or file path.")
# else:
#     print("LibriSpeech dev-clean already extracted.")


In [4]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
DATA_URL = "http://www.openslr.org/resources/12/train-clean-100.tar.gz" # Changed to train-clean-100
DATA_FOLDER = '/content/drive/MyDrive/LibriSpeech/'
tar_path = os.path.join(DATA_FOLDER, 'train-clean-100.tar.gz') # Changed to train-clean-100

In [6]:
# # Check if the extracted data exists in the Google Drive location
# extracted_data_path = os.path.join(DATA_FOLDER, 'train-clean-100')
# if not os.path.exists(extracted_data_path):
#     with tarfile.open(tar_path, 'r:gz') as tar:
#             print("Extracting files...")
#             # Use filter='data' for safer extraction
#             tar.extractall(path=DATA_FOLDER, filter='data')
#             print("Extraction complete.")

# else:
#     print("LibriSpeech train-clean-100 already extracted.")

In [7]:
def parse_librispeech_metadata(root_dir: str) -> pd.DataFrame:
    """Reads all TRANS.TXT files to create a single DataFrame of all samples."""
    all_data = []
    data_path = os.path.join(root_dir, 'LibriSpeech/dev-clean')

    if not os.path.exists(data_path):
        return pd.DataFrame()

    for dirpath, _, filenames in os.walk(data_path):
        for filename in filenames:
            if filename.endswith('.txt'):
                transcript_path = os.path.join(dirpath, filename)
                with open(transcript_path, 'r') as f:
                    for line in f:
                        line = line.strip()
                        if line:
                            audio_id, normalized_text = line.split(' ', 1)
                            normalized_text = normalized_text.upper().strip()
                            audio_path = os.path.join(dirpath, audio_id + '.flac')

                            if os.path.exists(audio_path):
                                all_data.append({
                                    'file_path': audio_path,
                                    'normalized_text': normalized_text
                                })

    return pd.DataFrame(all_data)

In [8]:
def parse_librispeech_metadata(root_dir: str) -> pd.DataFrame:
    """Reads all TRANS.TXT files to create a single DataFrame of all samples."""
    all_data = []
    data_path = os.path.join(root_dir, 'LibriSpeech/train-clean-100')

    if not os.path.exists(data_path):
        return pd.DataFrame()

    for dirpath, _, filenames in os.walk(data_path):
        for filename in filenames:
            if filename.endswith('.txt'):
              print('file found')
              transcript_path = os.path.join(dirpath, filename)
              with open(transcript_path, 'r') as f:
                  for line in f:
                      line = line.strip()
                      if line:
                          audio_id, normalized_text = line.split(' ', 1)
                          normalized_text = normalized_text.upper().strip()
                          audio_path = os.path.join(dirpath, audio_id + '.flac')
                          print(audio_path)
                          if os.path.exists(audio_path):
                              all_data.append({
                                  'file_path': audio_path,
                                  'normalized_text': normalized_text
                              })
    return pd.DataFrame(all_data)

In [ ]:
df_full = parse_librispeech_metadata(DATA_FOLDER)
print(f"Total samples parsed: {len(df_full)}")

In [61]:
df_full = df_full.sample(frac=1, random_state=42).reset_index(drop=True)
len(df_full)

28389

In [62]:
if not df_full.empty:
    df_train, df_val = train_test_split(df_full, test_size=0.1, random_state=42, shuffle=True)
    df_train, df_test = train_test_split(df_train, test_size=0.1, random_state=42, shuffle=True)
    print(f"Training samples: {len(df_train)}, Validation samples: {len(df_val)}, Test samples: {len(df_test)}")
else:
    df_train = pd.DataFrame()
    df_val = pd.DataFrame()
    print("Cannot split data: DataFrame is empty.")


Training samples: 22995, Validation samples: 2839, Test samples: 2555


# Vocabulary Creation

In [63]:
# 3. Vocabulary Creation
def create_vocab(df: pd.DataFrame) -> Tuple[Dict, Dict, int]:
    """Creates char-to-int and int-to-char maps."""
    if df.empty:
        default_chars = ['<PAD>', '<SOS>', '<EOS>', '<UNK>', 'A', '|']
        char_to_int = {c: i for i, c in enumerate(default_chars)}
        int_to_char_map = {i: c for i, c in enumerate(default_chars)}
        return char_to_int, int_to_char_map, len(default_chars)

    all_text = ' '.join(df['normalized_text'].tolist())
    unique_chars = sorted(list(set(all_text.replace(' ', '')) | set(['|'])))

    special_tokens = {'<PAD>': PAD_ID, '<SOS>': SOS_ID, '<EOS>': EOS_ID, '<UNK>': UNK_ID}
    char_to_int = special_tokens.copy()
    current_id = 4

    for char in unique_chars:
        if char not in char_to_int:
            char_to_int[char] = current_id
            current_id += 1

    int_to_char_map = {v: k for k, v in char_to_int.items()}
    vocab_size = len(char_to_int)

    return char_to_int, int_to_char_map, vocab_size

In [64]:
char_to_int, int_to_char_map, VOCAB_SIZE = create_vocab(df_train)
print(f"Vocabulary Size (Output Dimension): {VOCAB_SIZE}")


Vocabulary Size (Output Dimension): 32


# Feature extraction

## FEx

In [65]:
# --- Feature Extraction and Enhancement (using librosa) ---
def extract_and_enhance_features(waveform_np: np.ndarray, sample_rate: int = SAMPLE_RATE):
    n_mels = 40

    mel_spectrogram_db = librosa.power_to_db(
        librosa.feature.melspectrogram(
            y=waveform_np,
            sr=sample_rate,
            n_mels=n_mels,
            n_fft=400,
            hop_length=160
        ),
        ref=np.max
    )

    features = mel_spectrogram_db.T # (T, 40)

    deltas = librosa.feature.delta(features, width=9)
    double_deltas = librosa.feature.delta(deltas, width=9)

    enhanced_features_np = np.concatenate([features, deltas, double_deltas], axis=-1)

    mean = np.mean(enhanced_features_np, axis=0, keepdims=True)
    std = np.std(enhanced_features_np, axis=0, keepdims=True)
    normalized_features_np = (enhanced_features_np - mean) / (std + 1e-5)

    normalized_features = torch.from_numpy(normalized_features_np).float()

    return normalized_features

## CTC Dataset

In [66]:
import soundfile as sf
import torch
from torch.utils.data import Dataset
import numpy as np
import pandas as pd
from typing import Dict

# --- CTC Dataset (soundfile-based) ---
class CTCSpeechDataset(Dataset):
    def __init__(self, data_df: pd.DataFrame, char_to_int: Dict[str, int], sample_rate: int = 16000):
        self.data_df = data_df.reset_index(drop=True)
        self.char_to_int = char_to_int
        self.unk_id = char_to_int['<UNK>']
        self.sample_rate = sample_rate
        self._flen_cache = {}  # idx -> int(feature_length)

    def __len__(self):
        return len(self.data_df)

    def __getitem__(self, idx):
        row = self.data_df.iloc[idx]

        # --- load audio with soundfile ---
        waveform_np, file_sr = sf.read(row['file_path'], dtype="float32")

        # if stereo → convert to mono
        if waveform_np.ndim > 1:
            waveform_np = np.mean(waveform_np, axis=1)

        # optional resample
        if file_sr != self.sample_rate:
            import librosa
            waveform_np = librosa.resample(waveform_np, orig_sr=file_sr, target_sr=self.sample_rate)

        # --- extract features ---
        features = extract_and_enhance_features(waveform_np, self.sample_rate)

        # --- text to int sequence ---
        text = row['normalized_text'].replace(' ', '|')
        int_sequence = [self.char_to_int.get(c, self.unk_id) for c in text]
        labels = torch.tensor(int_sequence, dtype=torch.long)

        return {
            'features': features,
            'feature_length': int(features.size(0)),
            'labels': labels,
            'label_length': int(labels.size(0)),
        }


    def sort_by_transcript_length(self, reverse: bool = False, inplace: bool = True):
        """Sort by transcript (label) length."""
        order = np.argsort(self.data_df['normalized_text'].str.len().values)
        if reverse:
            order = order[::-1]

        sorted_df = self.data_df.iloc[order].reset_index(drop=True)

        if inplace:
            self.data_df = sorted_df
            return self
        else:
            return CTCSpeechDataset(sorted_df, self.char_to_int, self.sample_rate)


## CTC Collate Function

In [67]:
# --- CTC Collate Function ---
def collate_fn_ctc(batch: List[Dict]) -> Dict:
    max_feature_len = max(item['feature_length'] for item in batch)
    max_label_len = max(item['label_length'] for item in batch)
    batch_size = len(batch)

    features_padded = torch.zeros((batch_size, max_feature_len, INPUT_FEATURE_DIM))
    labels_padded = torch.zeros((batch_size, max_label_len), dtype=torch.long)
    feature_lengths = torch.tensor([item['feature_length'] for item in batch], dtype=torch.long)
    label_lengths = torch.tensor([item['label_length'] for item in batch], dtype=torch.long)

    for i, item in enumerate(batch):
        f_len = item['feature_length']
        features_padded[i, :f_len, :] = item['features']

        l_len = item['label_length']
        labels_padded[i, :l_len] = item['labels']

    return {
        'features': features_padded,
        'feature_lengths': feature_lengths,
        'labels': labels_padded,
        'label_lengths': label_lengths
    }

# Models

## Encoder (Bi-directional RNN)

In [68]:
class Encoder(nn.Module):
    def __init__(self, input_feature_dim: int, hidden_size: int, dropout: float):
        super(Encoder, self).__init__()
        self.rnn = nn.RNN(
            input_size=input_feature_dim,
            hidden_size=hidden_size,
            num_layers=2,
            dropout=dropout,
            bidirectional=True,
            batch_first=True
        )

    def forward(self, features: torch.Tensor, feature_lengths: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # Pack sequence to handle variable lengths
        packed_input = nn.utils.rnn.pack_padded_sequence(
            features, feature_lengths.cpu(), batch_first=True, enforce_sorted=False
        )

        # RNN returns (output, h_n)
        packed_output, h_n = self.rnn(packed_input)

        # Unpack sequence
        output_sequence, _ = nn.utils.rnn.pad_packed_sequence(
            packed_output, batch_first=True
        )

        # Return output sequence and None for compatibility with CTCLASR
        return output_sequence, None

## CTC Head

In [69]:
class CTCDecoderHead(nn.Module):
    def __init__(self, hidden_size: int, vocab_size: int):
        super(CTCDecoderHead, self).__init__()
        self.output_projection = nn.Linear(hidden_size*2, vocab_size)

    def forward(self, encoder_outputs: torch.Tensor) -> torch.Tensor:
        logits = self.output_projection(encoder_outputs)
        return F.log_softmax(logits, dim=-1)

class CTCLASR(nn.Module):
    def __init__(self, input_feature_dim: int, hidden_size: int, vocab_size: int, dropout: float = 0.1):
        super(CTCLASR, self).__init__()
        self.encoder = Encoder(input_feature_dim, hidden_size, dropout)
        self.ctc_head = CTCDecoderHead(hidden_size, vocab_size)

    def forward(self, features: torch.Tensor, feature_lengths: torch.Tensor) -> torch.Tensor:
        encoder_outputs, _ = self.encoder(features, feature_lengths)
        log_probs = self.ctc_head(encoder_outputs)
        return log_probs.transpose(0, 1)

# Training

### Training Epoch

In [70]:
def train_epoch(model: nn.Module, data_loader: DataLoader, optimizer: optim.Optimizer, criterion: nn.Module) -> float:
    model.train()
    total_loss = 0
    for i, batch in enumerate(data_loader):
        if i%5 == 0:
          print(f"Batch {i}/{len(data_loader)} Up for training")
        if not batch: continue
        features, feature_lengths = batch['features'].to(DEVICE), batch['feature_lengths'].to(DEVICE)
        labels, label_lengths = batch['labels'].to(DEVICE), batch['label_lengths'].to(DEVICE)
        optimizer.zero_grad()
        log_probs = model(features, feature_lengths)
        loss = criterion(log_probs=log_probs, targets=labels, input_lengths=feature_lengths, target_lengths=label_lengths)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(data_loader)

### Greedy Decode

In [71]:
def greedy_ctc_decode(log_probs: torch.Tensor, int_to_char_map: dict) -> List[str]:
    best_paths = torch.argmax(log_probs.transpose(0, 1).cpu(), dim=-1).numpy()
    decoded_texts = []
    for path in best_paths:
        decoded_sequence = []
        last_char = BLANK_ID
        for char_id in path:
            if char_id != BLANK_ID and char_id != last_char:
                decoded_sequence.append(char_id)
            last_char = char_id
        decoded_text = "".join([int_to_char_map.get(id, '') for id in decoded_sequence])
        decoded_texts.append(decoded_text.replace('|', ' '))
    return decoded_texts

In [72]:
def calculate_wer_cer(reference: str, hypothesis: str) -> Tuple[float, float]:
    ref_words, hyp_words = reference.split(), hypothesis.split()
    word_error = editdistance.eval(ref_words, hyp_words)
    wer = word_error / len(ref_words) if ref_words else 0.0
    char_error = editdistance.eval(reference, hypothesis)
    cer = char_error / len(reference) if reference else 0.0
    return wer, cer

### Evaluate

In [73]:
def evaluate_model(model: nn.Module, data_loader: DataLoader, criterion: nn.Module, int_to_char_map: dict, print_examples: int = 5) -> Tuple[float, float, float]:
    model.eval()
    total_loss, total_wer, total_cer, total_samples = 0, 0, 0, 0
    examples_printed = 0
    print("\n--- Starting Evaluation ---")
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            if i%5 == 0:
              print(f"Batch {i}/{len(data_loader)} Up for eval")
            if not batch: continue
            features, feature_lengths = batch['features'].to(DEVICE), batch['feature_lengths'].to(DEVICE)
            labels, label_lengths = batch['labels'].to(DEVICE), batch['label_lengths'].to(DEVICE)
            log_probs = model(features, feature_lengths)
            loss = criterion(log_probs=log_probs, targets=labels, input_lengths=feature_lengths, target_lengths=label_lengths)
            total_loss += loss.item()
            predicted_texts = greedy_ctc_decode(log_probs, int_to_char_map)

            for j in range(features.size(0)):
                label_sequence = labels[j, :label_lengths[j]].tolist()
                reference = "".join([int_to_char_map.get(id, '') for id in label_sequence]).replace('|', ' ')
                hypothesis = predicted_texts[j]
                if not reference: continue
                wer, cer = calculate_wer_cer(reference, hypothesis)
                total_wer += wer
                total_cer += cer
                total_samples += 1

                if examples_printed < print_examples:
                    print("-" * 50)
                    print(f"Sample {total_samples}")
                    print(f"  REFERENCE (R): {reference}")
                    print(f"  HYPOTHESIS (H): {hypothesis}")
                    print(f"  WER: {wer*100:.2f}%, CER: {cer*100:.2f}%")
                    examples_printed += 1

    avg_loss = total_loss / len(data_loader)
    avg_wer = (total_wer / total_samples) * 100
    avg_cer = (total_cer / total_samples) * 100

    print("\n" + "=" * 50)
    print("--- Evaluation Metrics ---")
    print(f"Average CTC Loss: {avg_loss:.4f}")
    print(f"Average Word Error Rate (WER): {avg_wer:.2f}%")
    print(f"Average Character Error Rate (CER): {avg_cer:.2f}%")
    print("=" * 50)
    return avg_loss, avg_wer, avg_cer

In [74]:
def train_model(model, train_loader, val_loader, optimizer, criterion, num_epochs, int_to_char_map, model_save_path: str = "best_ctc_model.pt"):
    print(f"\nStarting training on device: {DEVICE}")
    best_val_wer = float('inf')
    train_losses, val_losses, val_wers = [], [], []

    for epoch in range(1, num_epochs + 1):
        print(f"\n--- Epoch {epoch}/{num_epochs} Starting ---")
        start_time = time.time()
        train_loss = train_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_wer, val_cer = evaluate_model(model, val_loader, criterion, int_to_char_map, print_examples=5)
        end_time = time.time()

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_wers.append(val_wer)

        print(f"Epoch {epoch}/{num_epochs} Done | Time: {end_time - start_time:.2f}s")
        print(f"  Train Loss: {train_loss:.4f} | Valid Loss: {val_loss:.4f}")
        print(f"  Valid WER: {val_wer:.2f}% | Valid CER: {val_cer:.2f}%")

        if val_wer < best_val_wer:
            best_val_wer = val_wer
            torch.save(model.state_dict(), model_save_path)
            print(f"  Model Saved! New Best Validation WER: {best_val_wer:.2f}%")

    print("\nTraining complete.")
    return train_losses, val_losses, val_wers

In [75]:

def plot_metrics(train_losses, val_losses, val_wers):
    epochs = range(1, len(train_losses) + 1)
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_losses, 'b-o', label='Training Loss')
    plt.plot(epochs, val_losses, 'r-o', label='Validation Loss')
    plt.title('Training and Validation Loss (CTC)')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, val_wers, 'g-o', label='Validation WER')
    plt.title('Validation Word Error Rate (WER)')
    plt.xlabel('Epochs')
    plt.ylabel('WER (%)')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig('training_metrics.png')
    plt.show()
    print("\n has been generated.")



# Execution

In [76]:
# --- HYPERPARAMETERS ---
HIDDEN_SIZE = 512
BATCH_SIZE = 32
NUM_EPOCHS = 10
LEARNING_RATE = 0.001

In [77]:
train_dataset = CTCSpeechDataset(df_train, char_to_int).sort_by_transcript_length(inplace=True)
val_dataset = CTCSpeechDataset(df_val, char_to_int).sort_by_transcript_length(inplace=True)
test_dataset = CTCSpeechDataset(df_test, char_to_int)

In [78]:
train_loader = DataLoader(
    dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate_fn_ctc
)
val_loader = DataLoader(
    dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_fn_ctc
)
test_loader = DataLoader(
    dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_fn_ctc
)

In [80]:
model = CTCLASR(
    input_feature_dim=INPUT_FEATURE_DIM,
    hidden_size=HIDDEN_SIZE,
    vocab_size=VOCAB_SIZE,
    dropout=0.1
).to(DEVICE)

criterion = nn.CTCLoss(blank=BLANK_ID, zero_infinity=True).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-3)

In [ ]:
if not df_train.empty and not df_val.empty:
    metrics = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        criterion=criterion,
        num_epochs=NUM_EPOCHS,
        int_to_char_map=int_to_char_map
    )

    train_losses, val_losses, val_wers = metrics
else:
    print("\n[FINAL CHECK] Cannot start training. DataFrames are empty. Check download and parsing steps.")

In [ ]:
plot_metrics(train_losses, val_losses, val_wers)

In [ ]:
def test_model(model: nn.Module, data_loader: DataLoader, criterion: nn.Module, int_to_char_map: dict, print_examples: int = 10) -> Tuple[float, float, float]:
    """
    Run evaluation on test data and print detailed outputs.

    Args:
        model: trained ASR model (e.g., CTCLASR)
        data_loader: test DataLoader
        criterion: CTC loss
        int_to_char_map: mapping from int -> character
        print_examples: number of test samples to print

    Returns:
        avg_loss, avg_wer, avg_cer
    """
    model.eval()
    total_loss, total_wer, total_cer, total_samples = 0, 0, 0, 0
    examples_printed = 0

    print("\n--- Starting Testing ---")
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            if not batch:
                continue
            features, feature_lengths = batch['features'].to(DEVICE), batch['feature_lengths'].to(DEVICE)
            labels, label_lengths = batch['labels'].to(DEVICE), batch['label_lengths'].to(DEVICE)

            # Forward pass
            log_probs = model(features, feature_lengths)
            loss = criterion(log_probs=log_probs, targets=labels,
                             input_lengths=feature_lengths,
                             target_lengths=label_lengths)
            total_loss += loss.item()

            # Decode predictions
            predicted_texts = greedy_ctc_decode(log_probs, int_to_char_map)

            # Compare each sample
            for j in range(features.size(0)):
                label_sequence = labels[j, :label_lengths[j]].tolist()
                reference = "".join([int_to_char_map.get(idx, '') for idx in label_sequence]).replace('|', ' ')
                hypothesis = predicted_texts[j]

                if not reference:
                    continue

                wer, cer = calculate_wer_cer(reference, hypothesis)
                total_wer += wer
                total_cer += cer
                total_samples += 1

                if examples_printed < print_examples:
                    print("-" * 60)
                    print(f"Sample {total_samples}")
                    print(f"  REFERENCE (R): {reference}")
                    print(f"  HYPOTHESIS (H): {hypothesis}")
                    print(f"  WER: {wer*100:.2f}%, CER: {cer*100:.2f}%")
                    examples_printed += 1

    avg_loss = total_loss / len(data_loader)
    avg_wer = (total_wer / total_samples) * 100
    avg_cer = (total_cer / total_samples) * 100

    print("\n" + "=" * 50)
    print("--- Test Metrics ---")
    print(f"Average CTC Loss: {avg_loss:.4f}")
    print(f"Average Word Error Rate (WER): {avg_wer:.2f}%")
    print(f"Average Character Error Rate (CER): {avg_cer:.2f}%")
    print("=" * 50)

    return avg_loss, avg_wer, avg_cer


In [ ]:
test_loss, test_wer, test_cer = test_model(
    model,
    test_loader
    criterion,
    int_to_char_map,
    print_examples=10
)
